# **Introduction**

Mental health has become a critical global concern, with over 970 million people suffering from conditions such as depression, anxiety, and stress-related disorders. Despite this, access to professional care remains limited due to factors like stigma, cost, and shortage of trained specialists, especially in low-income or rural regions. Many individuals avoid seeking help because of social judgment or lack of resources, which leads to untreated symptoms and declining well-being.

To address these challenges, digital health technologies are emerging as scalable solutions. In particular, Natural Language Processing (NLP) and Conversational AI have shown potential to provide affordable, accessible, and empathetic support. Conversational agents (chatbots or virtual counselors) can simulate therapeutic dialogue, offering users an anonymous and non-judgmental space to share emotions and receive guidance.

However, most existing systems rely on rule-based or retrieval-based approaches, which produce rigid or generic responses and often fail to recognize users’ emotional needs. Without ethical safeguards, they risk giving insensitive, harmful, or misleading advice, especially in crisis situations.

Therefore, there is an urgent need to design emotion-aware and ethically responsible conversational AI systems. By leveraging large language models (LLMs), emotion and intent detection, and multi-layered ethical filters, such systems can provide empathetic, safe, and contextually appropriate conversations. This research focuses on building such a system to bridge the gap between technology and psychological support, contributing to both academic advancement in NLP and practical applications in digital mental health.

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


lets you access, read, and save files directly between Colab and your Drive.

In [3]:
!pip install transformers torch pandas scikit-learn sentencepiece accelerate -q

This code sets up the essential tools for building and training NLP models for mental health conversations. It uses Pandas for handling datasets and Regex (re) for text preprocessing, while scikit-learn’s train_test_split is applied to divide data into training and testing sets. The PyTorch (torch) library provides the deep learning framework to train models. From Hugging Face’s Transformers, GPT-2 is imported for conversational response generation, and DistilBERT is imported for sequence classification tasks like intent detection. Supporting utilities such as Trainer, TrainingArguments, and DataCollatorForLanguageModeling simplify fine-tuning, batching, and managing experiments. Finally, accuracy_score and f1_score are used as evaluation metrics to measure how well the models perform.

In [4]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import re
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


This code is designed to safely load the dataset required for training. It tries to read the file **`train.csv`** from Google Drive using Pandas. If the file is found, it confirms successful loading and displays the first few rows of the dataset as a preview. However, if the file is missing, it catches the `FileNotFoundError` and prints an error message instead of crashing. In that case, it creates an empty DataFrame so the program can still run without immediately failing. This makes the code more reliable and user-friendly.


In [6]:
try:
    df = pd.read_csv('/content/drive/MyDrive/train.csv')
    print("Dataset loaded successfully.")
    print("Dataset preview:")
    print(df.head())
except FileNotFoundError:
    print("Error: 'train.csv' not found. Please upload your dataset.")
    df = pd.DataFrame()

Dataset loaded successfully.
Dataset preview:
                                             Context  \
0  I'm going through some things with my feelings...   
1  I'm going through some things with my feelings...   
2  I'm going through some things with my feelings...   
3  I'm going through some things with my feelings...   
4  I'm going through some things with my feelings...   

                                            Response  
0  If everyone thinks you're worthless, then mayb...  
1  Hello, and thank you for your question and see...  
2  First thing I'd suggest is getting the sleep y...  
3  Therapy is essential for those that are feelin...  
4  I first want to let you know that you are not ...  


This code displays the dataset size before and after cleaning. Initially, the dataset contained 3512 rows, but after removing rows with missing values in the ‘Context’ or ‘Response’ columns, the size reduced to 3508 rows. The index was then reset for consistency. This ensures the dataset is clean and reliable for training NLP models.

In [7]:
print(f"Original dataset size: {len(df)}")
df.dropna(subset=['Context', 'Response'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Dataset size after removing nulls: {len(df)}")

Original dataset size: 3512
Dataset size after removing nulls: 3508


This code defines a **text normalization function** to clean and standardize the dataset. It converts text to lowercase, removes extra spaces, and filters out unwanted characters (keeping only letters, numbers, spaces, and punctuation like `. , ? !`). The function is then applied to both **Context** (user input) and **Response** (therapist reply) columns, creating cleaned versions: `Context_cleaned` and `Response_cleaned`. The output shows neatly processed text, which is essential for training NLP models to understand language consistently.


In [8]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s.,?!]', '', text)
    return text.strip()
print("\nNormalizing text...")
df['Context_cleaned'] = df['Context'].apply(normalize_text)
df['Response_cleaned'] = df['Response'].apply(normalize_text)
print("Text normalization complete.")
print(df[['Context_cleaned', 'Response_cleaned']].head())


Normalizing text...
Text normalization complete.
                                     Context_cleaned  \
0  im going through some things with my feelings ...   
1  im going through some things with my feelings ...   
2  im going through some things with my feelings ...   
3  im going through some things with my feelings ...   
4  im going through some things with my feelings ...   

                                    Response_cleaned  
0  if everyone thinks youre worthless, then maybe...  
1  hello, and thank you for your question and see...  
2  first thing id suggest is getting the sleep yo...  
3  therapy is essential for those that are feelin...  
4  i first want to let you know that you are not ...  


This code initializes the **GPT-2 Medium model** along with its tokenizer from Hugging Face’s Transformers library. The tokenizer is responsible for breaking down input text into tokens that the model can understand, while the GPT-2 language model is used for generating context-aware responses. By sending the model to the appropriate device (`GPU` if available, otherwise `CPU`), the setup ensures efficient computation. During this process, the necessary configuration and weight files (such as vocabulary, merges, and model parameters) are automatically downloaded. The warning about the Hugging Face token is optional, as authentication is only required for private models—public models like GPT-2 can still be accessed without it. This step prepares the base model for **fine-tuning on the mental health conversation dataset** so it can learn to generate empathetic, therapist-like replies.


In [9]:
model_name = 'gpt2-medium'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

This code prepares the mental health conversation data for fine-tuning GPT-2. Since GPT-2 does not have a default padding token, the pad token is set to the end-of-sequence token to handle variable-length inputs during training. Each dataset row is then reformatted into a structured chat format where the **context** (user’s input) and the **response** (therapist’s reply) are combined with special tokens, making it easier for the model to learn conversation flow. A custom PyTorch dataset class `MentalHealthDataset` is created to tokenize the data, pad or truncate sequences to a fixed length, and return both the inputs and their corresponding labels. By setting the labels equal to the input IDs, the model is trained in a language modeling setup, enabling it to predict therapist-like responses given a user’s input.


In [10]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
df['chat_formatted'] = df.apply(lambda row: f"<|startoftext|>Context: {row['Context_cleaned']}\nResponse: {row['Response_cleaned']}<|endoftext|>", axis=1)

class MentalHealthDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_length)

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = item['input_ids'].clone()
        return item

This code snippet is part of preparing data for training a machine learning model, likely in NLP. First, the `train_test_split` function from `scikit-learn` is used to split the `chat_formatted` column of the DataFrame `df` into two sets: `train_texts` for training and `val_texts` for validation. The `test_size=0.1` argument specifies that 10% of the data will be used for validation, while the remaining 90% will be used for training. The `random_state=42` ensures that the split is reproducible, meaning the same split will occur every time the code is run. Next, these text lists are wrapped into custom dataset objects, `train_dataset` and `val_dataset`, using the `MentalHealthDataset` class. This class likely handles tokenization of the texts using the provided `tokenizer` and converts them into a format suitable for model training, such as PyTorch tensors, allowing the data to be efficiently fed into an NLP model during training and validation.


In [11]:
train_texts, val_texts = train_test_split(df['chat_formatted'].tolist(), test_size=0.1, random_state=42)

train_dataset = MentalHealthDataset(train_texts, tokenizer)
val_dataset = MentalHealthDataset(val_texts, tokenizer)

This code sets up the training configuration for a Hugging Face `Trainer` using the `TrainingArguments` class. It specifies that the trained model and checkpoints will be saved in the `./results_gpt2` directory, and training will run for 4 epochs. The batch size for both training and evaluation is set to 4 per device. To stabilize training, the learning rate will gradually warm up over the first 500 steps, and a weight decay of 0.01 is applied to reduce overfitting. Training metrics will be logged every 100 steps in the `./logs_gpt2` directory. The model will be evaluated and checkpointed at the end of each epoch, and after training, the best-performing model on the validation set will be automatically loaded. Overall, these arguments control the learning schedule, regularization, logging, checkpointing, and evaluation behavior for efficient and reliable model training.


In [12]:
training_args = TrainingArguments(
    output_dir='./results_gpt2',
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_gpt2',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

This code creates a `data_collator` using Hugging Face’s `DataCollatorForLanguageModeling`, which is responsible for dynamically preparing batches of data during training. By passing the `tokenizer`, it ensures that input texts are properly converted into token IDs and padded to the same length within each batch. The argument `mlm=False` indicates that the model is being trained for **causal language modeling** (like GPT-style models) rather than **masked language modeling** (used in BERT), so no tokens are randomly masked for prediction. Essentially, this `data_collator` automates batching and formatting the text data in a way the model can efficiently process during training.


In [13]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

This code initializes a Hugging Face `Trainer` object, which handles the training and evaluation of a language model. The `model` argument specifies the pre-trained model to be fine-tuned. The `args` argument passes the previously defined `training_args`, which control the training configuration such as batch size, number of epochs, logging, and checkpointing. The `train_dataset` and `eval_dataset` arguments provide the training and validation datasets, respectively, wrapped in the custom `MentalHealthDataset` class. Finally, `data_collator` ensures that the input texts are properly tokenized, padded, and formatted into batches suitable for causal language modeling. Together, this `Trainer` setup automates the entire training loop, including forward and backward passes, evaluation, logging, and saving model checkpoints.


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

The output shows the complete process of fine-tuning GPT-2 using Hugging Face’s `Trainer`. The script begins with a message indicating the start of fine-tuning, followed by a harmless warning from the notebook environment about an invalid escape sequence. Hugging Face then integrates with Weights & Biases (wandb) for experiment tracking, where you logged into your wandb account and the run details were synced online. The model was trained for four epochs, during which both training and validation losses consistently decreased, showing that the model was effectively learning patterns from the dataset. Training loss improved from 2.57 in the first epoch to 1.55 in the last, while validation loss decreased from 2.40 to 1.90, reflecting good generalization without severe overfitting. A checkpoint warning about a missing key (`lm_head.weight`) appeared, which is common when initializing GPT-2 and was handled automatically by the trainer. Once training was complete, the script printed a confirmation message and saved the fine-tuned model along with the tokenizer to the directory `./fine_tuned_gpt2_mental_health`, making it ready for further evaluation or real-world use.


In [15]:
print("\nStarting GPT-2 fine-tuning...")
trainer.train()
print("Fine-tuning complete.")
final_model_path = './fine_tuned_gpt2_mental_health'
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Model saved to {final_model_path}")


Starting GPT-2 fine-tuning...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: s-s-savardekar2 (s-s-savardekar2-newcastle-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.576800,2.402541
2,2.114600,2.086995
3,1.690500,1.947164


Epoch,Training Loss,Validation Loss
1,2.576800,2.402541
2,2.114600,2.086995
3,1.690500,1.947164
4,1.554000,1.909534


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Fine-tuning complete.
Model saved to ./fine_tuned_gpt2_mental_health


In this code, a simple rule-based intent classifier is created to assign placeholder intent labels to text data. The `intent_map` defines five intent categories: *Seek Support*, *Express Emotion*, *Request Guidance*, *Reflect*, and *Escalate to Crisis*. The function `get_placeholder_intent()` processes each text by converting it to lowercase and checking for the presence of specific keywords. If the text contains words like “help” or “advice,” it is labeled as *Request Guidance*; if it contains severe phrases like “suicide” or “can’t go on,” it is labeled as *Escalate to Crisis*; if it includes words such as “feel” or “I’m,” it is labeled as *Express Emotion*; and if it refers to past experiences with phrases like “remember” or “used to,” it is labeled as *Reflect*. Texts without these keywords default to *Seek Support*. When applied to the `Context_cleaned` column of the dataset, the function generated a new column `intent_label`. In the output, all five sample rows were labeled as *Escalate to Crisis*, suggesting that the function detected crisis-related keywords across those examples, though it may be overly sensitive in its current form.


In [16]:
intent_map = {
    0: "Seek Support",
    1: "Express Emotion",
    2: "Request Guidance",
    3: "Reflect",
    4: "Escalate to Crisis"
}

def get_placeholder_intent(text):
    text = text.lower()
    if any(keyword in text for keyword in ["help", "advice", "what should i do", "guide me"]):
        return 2
    if any(keyword in text for keyword in ["suicide", "kill myself", "can't go on"]):
        return 4
    if any(keyword in text for keyword in ["feel", "feeling", "i am", "i'm"]):
        return 1
    if any(keyword in text for keyword in ["thinking about", "remember", "used to"]):
        return 3
    return 0
df['intent_label'] = df['Context_cleaned'].apply(get_placeholder_intent)
print("\nPlaceholder intent labels generated:")
print(df[['Context_cleaned', 'intent_label']].head())


Placeholder intent labels generated:
                                     Context_cleaned  intent_label
0  im going through some things with my feelings ...             4
1  im going through some things with my feelings ...             4
2  im going through some things with my feelings ...             4
3  im going through some things with my feelings ...             4
4  im going through some things with my feelings ...             4


In this code, we set up **DistilBERT** for intent classification on our dataset. We begin by loading a pre-trained tokenizer and a sequence classification model, specifying the number of labels according to our `intent_map`. The cleaned text data (`Context_cleaned`) and their intent labels (`intent_label`) are then split into training and validation sets using stratified sampling to maintain class balance. To make the data compatible with the model, we define a custom PyTorch dataset class called `IntentDataset`, which links tokenized inputs with their labels and returns them as tensors. Both training and validation texts are tokenized with truncation, padding, and a maximum sequence length of 128 tokens, after which we create `train_dataset_intent` and `val_dataset_intent` for training and evaluation. The output confirms that the DistilBERT tokenizer and model weights were downloaded, but the classification layers were newly initialized, which means the model must be fine-tuned on our intent dataset to learn meaningful representations and make accurate predictions.


In [17]:
intent_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
intent_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(intent_map)).to(device)

X = df['Context_cleaned'].tolist()
y = df['intent_label'].tolist()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_encodings = intent_tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = intent_tokenizer(X_val, truncation=True, padding=True, max_length=128)

train_dataset_intent = IntentDataset(train_encodings, y_train)
val_dataset_intent = IntentDataset(val_encodings, y_val)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In this code, we define a function `compute_metrics()` that will be used to evaluate our intent classification model during training and validation. The function takes in `pred`, which contains the model’s predictions and the true labels. First, it extracts the ground-truth labels using `pred.label_ids`. Then, it processes the model’s raw prediction outputs (`pred.predictions`) by applying `argmax(-1)` to select the class with the highest probability for each example. After obtaining the predicted labels, the function calculates two evaluation metrics: **F1-score** and **accuracy**. The F1-score is computed with a *weighted* average, meaning it considers class imbalance by weighting each class’s contribution according to its support in the dataset. The accuracy is the proportion of correct predictions over the total predictions. Finally, the function returns both metrics as a dictionary, making them available for reporting and logging during the training process.


In [18]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

Here, we trained our **DistilBERT intent classification model** using Hugging Face’s `Trainer`. The `TrainingArguments` specified that training would run for 3 epochs with a batch size of 16 for both training and evaluation, while logging, evaluation, and saving checkpoints were set to occur once per epoch. The argument `load_best_model_at_end=True` ensured that the model with the best validation performance was kept after training.

During training, the model showed strong performance improvements. In the first epoch, the training loss dropped to **0.2991**, with a validation loss of **0.1761**, and it achieved an accuracy of **96.15%** with a weighted F1-score of **95.63%**. By the second epoch, training loss fell further to **0.0483**, validation loss improved to **0.0605**, and both accuracy and F1-score reached approximately **98.86%**, showing excellent generalization. In the third epoch, the model stabilized with a training loss of **0.0252** and a validation loss of **0.0529**, maintaining the same high accuracy and F1-score.

Finally, the message *“Intent model training complete.”* confirmed the successful completion of the training process. This shows that our fine-tuned DistilBERT model has learned to classify intents with very high accuracy and is now ready for inference.


In [19]:
intent_training_args = TrainingArguments(
    output_dir='./results_intent',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs_intent',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

intent_trainer = Trainer(
    model=intent_model,
    args=intent_training_args,
    train_dataset=train_dataset_intent,
    eval_dataset=val_dataset_intent,
    compute_metrics=compute_metrics,
)

print("\nStarting intent classification model training...")
intent_trainer.train()
print("Intent model training complete.")


Starting intent classification model training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.299100,0.176135,0.961538,0.956359
2,0.048300,0.060454,0.988604,0.988441
3,0.025200,0.052906,0.988604,0.988441


Intent model training complete.


In this step, we saved our fine-tuned intent classification model and its tokenizer. The line `intent_trainer.save_model(final_intent_model_path)` stores the trained DistilBERT model, including its updated weights, into the directory `./distilbert_intent_classifier`. Next, `intent_tokenizer.save_pretrained(final_intent_model_path)` saves the tokenizer configuration and vocabulary to the same directory, ensuring that future inference uses the exact same tokenization process as during training. Finally, the confirmation message *“Intent model saved to ./distilbert\_intent\_classifier”* verifies that both the model and tokenizer have been successfully stored and are now ready to be reloaded for predictions, evaluation, or deployment.


In [20]:
final_intent_model_path = './distilbert_intent_classifier'
intent_trainer.save_model(final_intent_model_path)
intent_tokenizer.save_pretrained(final_intent_model_path)
print(f"Intent model saved to {final_intent_model_path}")

Intent model saved to ./distilbert_intent_classifier


In [21]:
base_path = "/content/drive/MyDrive/train.csv"

df = pd.read_csv(f"/content/drive/MyDrive/train.csv")


This code reloads our two fine-tuned models for inference. The GPT-2 response generation model and tokenizer are loaded from `./fine_tuned_gpt2_mental_health`, while the DistilBERT intent classification model and tokenizer are loaded from `./distilbert_intent_classifier`. The device is set to GPU if available, otherwise CPU. An `intent_map` is also defined to map numeric predictions to intent labels. With both models successfully loaded, we can now classify intents and generate responses.


In [22]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DistilBertTokenizer, DistilBertForSequenceClassification
import torch

final_model_path = './fine_tuned_gpt2_mental_health'
final_intent_model_path = './distilbert_intent_classifier'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading the response generation model...")
response_tokenizer = GPT2Tokenizer.from_pretrained(final_model_path)
response_model = GPT2LMHeadModel.from_pretrained(final_model_path).to(device)
print("Response generation model loaded successfully.")

print("\nLoading the intent classification model...")
intent_tokenizer = DistilBertTokenizer.from_pretrained(final_intent_model_path)
intent_model = DistilBertForSequenceClassification.from_pretrained(final_intent_model_path).to(device)
print("Intent classification model loaded successfully.")

intent_map = {
    0: "Seek Support",
    1: "Express Emotion",
    2: "Request Guidance",
    3: "Reflect",
    4: "Escalate to Crisis"
}

Loading the response generation model...
Response generation model loaded successfully.

Loading the intent classification model...
Intent classification model loaded successfully.


This code defines three core functions that drive our chatbot system. The `normalize_text()` function standardizes user input by converting it to lowercase, removing unnecessary characters, collapsing extra spaces, and trimming the text, ensuring consistency before further processing. The `classify_intent()` function tokenizes the cleaned input with the DistilBERT tokenizer, passes it through the intent classification model, and retrieves the predicted class ID by selecting the label with the highest score. The `generate_response()` function prepares a prompt that includes the user input, encodes it, and feeds it into the fine-tuned GPT-2 model, which generates a context-aware response under constraints such as maximum length, no repeated n-grams, and early stopping. The output is then decoded, and only the part after “Response:” is extracted and returned as the chatbot’s reply.


In [23]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s.,?!]', '', text)
    return text.strip()

def classify_intent(text):

    inputs = intent_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = intent_model(**inputs)
        logits = outputs.logits

    predicted_class_id = torch.argmax(logits, dim=1).item()
    return predicted_class_id

def generate_response(user_input):

    prompt = f"<|startoftext|>Context: {user_input}\nResponse:"

    inputs = response_tokenizer.encode(prompt, return_tensors='pt').to(device)


    outputs = response_model.generate(
        inputs,
        max_length=150,
        num_return_sequences=1,
        pad_token_id=response_tokenizer.eos_token_id,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    response_text = response_tokenizer.decode(outputs[0], skip_special_tokens=True)

    response_only = response_text.split("Response:")[1].strip()

    return response_only

This function defines the main chatbot interaction loop for providing mental health support. When we run `chat()`, it begins by displaying a warm introduction, explaining its supportive role and reminding the user that in cases of crisis they should contact a helpline. The chatbot then continuously waits for user input until the word *exit* is typed, at which point it politely ends the conversation. Each input is first cleaned with the `normalize_text()` function to ensure consistency. The intent is then classified using `classify_intent()`, and the corresponding label is displayed. If the detected intent indicates crisis (ID 4), the chatbot responds immediately with empathetic and urgent instructions, including hotline resources. For all other intents, the system attempts to generate a context-aware response using the GPT-2 model, ensuring that the user receives a supportive and relevant reply. If an error occurs during generation, the chatbot gracefully handles it by asking the user to rephrase. This loop ensures continuous, safe, and adaptive conversation flow.


In [24]:
def chat():

    print("\n--- Mental Health Support Chatbot ---")
    print("Hello! I'm here to listen and support you. Type 'exit' anytime to end our conversation.")
    print("If you are in a crisis, please call a helpline immediately.")
    print("-" * 40)

    while True:
        user_input = input("You: ")

        if user_input.lower() == 'exit':
            print("Chatbot: Thank you for talking with me. Take care.")
            break

        cleaned_input = normalize_text(user_input)

        intent_id = classify_intent(cleaned_input)
        intent_name = intent_map.get(intent_id, "Unknown Intent")
        print(f"(Detected Intent: {intent_name})")

        if intent_id == 4:
            print("\nChatbot: It sounds like you are in significant distress. It is very important to get immediate help. Please consider reaching out to a crisis hotline or emergency services. Here are some resources:\n- Crisis Text Line: Text HOME to 741741\n- National Suicide Prevention Lifeline: 988")
            continue

        try:
            bot_response = generate_response(cleaned_input)
            print(f"Chatbot: {bot_response}")
        except Exception as e:
            print(f"Chatbot: I'm sorry, I had trouble generating a response. Could you please rephrase? (Error: {e})")



we define the program’s entry point. When the script is launched, the chat() function starts automatically, initiating the conversation loop. However, if the file is imported elsewhere (for example, in another Python project), the chatbot will not start unintentionally, allowing us to reuse functions like normalize_text, classify_intent, or generate_response independently. This makes the code modular, safe, and more reusable.

In [25]:
if __name__ == '__main__':
    chat()


--- Mental Health Support Chatbot ---
Hello! I'm here to listen and support you. Type 'exit' anytime to end our conversation.
If you are in a crisis, please call a helpline immediately.
----------------------------------------
You: hii i m siddhi and i m suufering from mental health problem what should do


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


(Detected Intent: Request Guidance)
Chatbot: hi there, im so sorry to hear about this. there are so many different ways to deal with mental illness, and its important to find the right one for you. i would encourage you to talk with a local mentalhealth professional about your concerns and concerns about sufering. if you are concerned about stealing, i recommend that you talk to someone about that first. stealing is a very serious offense, so it is important that the person you speak to know that stealing will not be tolerated. you can always call 911 if theres a lot of commotion or if someone is in immediate
You: i need some guide from you 
(Detected Intent: Seek Support)
Chatbot: hello, and thank you for your question. i am very sorry that you lost your friend. it is very difficult to lose someone who we love so much. there are many things that one can do to help. first, i would like to commend you on taking the time to reach out to find out about a support group for friends and fami